# Capstone: a validated expense pipeline

**Stage:** Capstone · **Suggested time:** 45–90 minutes (capstone: several sessions)

**Prerequisites:** Concepts from notebooks 00–22. No variables or files from other notebooks are required.

## What you will learn

- Validate and quarantine malformed records
- Persist exact amounts to SQLite
- Produce and test a report from reopened data


## Your learning loop

1. Read the explanation and predict the next output.
2. Run the example with **Shift+Enter**.
3. Follow **Try a change**; explain the result in your own words.
4. Complete both exercises and enable their checks.
5. Build the mini-project, then use **Restart Kernel and Run All**.

Examples run as provided. Exercise starters are deliberately unfinished; their checks are disabled until you enable them. A successful Run All with checks disabled does **not** mean the exercises are complete.

Save a copy before extensive experiments. Use one notebook at a time to avoid mixing kernel state.


## 1. Define the domain and validate input

Build a complete small pipeline: CSV → validated records → SQLite → category report → CSV export. Store amounts as integer minor units: 12.50 becomes 1250. Invalid records should have a clear reason and must not enter storage. The fixture includes two invalid rows.


In [1]:
from dataclasses import dataclass
from datetime import date
from decimal import Decimal, InvalidOperation
import csv
import io

@dataclass(frozen=True)
class Expense:
    day: date
    category: str
    amount_minor: int

def parse_expense(row):
    try:
        day = date.fromisoformat(row["date"])
        category = row["category"].strip().lower()
        amount = Decimal(row["amount"])
    except (KeyError, TypeError, AttributeError, ValueError, InvalidOperation) as error:
        raise ValueError("missing or malformed field") from error
    if not category:
        raise ValueError("category must not be empty")
    if not amount.is_finite() or amount <= 0 or amount > Decimal("1000000000"):
        raise ValueError("amount must be finite, positive, and at most 1 billion")
    minor = amount * 100
    if minor != minor.to_integral_value():
        raise ValueError("amount must have at most two decimal places")
    return Expense(day, category, int(minor))

def load_expenses(csv_text):
    accepted, rejected = [], []
    for row_number, row in enumerate(csv.DictReader(io.StringIO(csv_text)), start=2):
        try:
            accepted.append(parse_expense(row))
        except ValueError as error:
            rejected.append({"row": row_number, "reason": str(error)})
    return accepted, rejected

csv_text = "date,category,amount\n2025-01-02, Food ,120.50\n2025-01-03,travel,80.00\n2025-01-04,food,49.50\n2025-02-01,books,200.00\ninvalid,food,10\n2025-01-06,food,-5\n"
expenses, rejected = load_expenses(csv_text)
print("Accepted:", expenses)
print("Rejected:", rejected)
assert len(expenses) == 4 and len(rejected) == 2


Accepted: [Expense(day=datetime.date(2025, 1, 2), category='food', amount_minor=12050), Expense(day=datetime.date(2025, 1, 3), category='travel', amount_minor=8000), Expense(day=datetime.date(2025, 1, 4), category='food', amount_minor=4950), Expense(day=datetime.date(2025, 2, 1), category='books', amount_minor=20000)]
Rejected: [{'row': 6, 'reason': 'missing or malformed field'}, {'row': 7, 'reason': 'amount must be finite, positive, and at most 1 billion'}]


**Try a change:** Add an amount of 1.005 and observe rejection. Why not silently round it?


## 2. Persist and reopen

Create a real temporary SQLite file, insert the validated rows in one transaction, close it, then reopen for the report. This demonstrates persistence across connections. Each run gets a new database, so rerunning does not duplicate the sample records.


In [2]:
import sqlite3
from contextlib import closing
from pathlib import Path
from tempfile import TemporaryDirectory

def save_expenses(connection, records):
    connection.execute("CREATE TABLE IF NOT EXISTS expenses (id INTEGER PRIMARY KEY, day TEXT NOT NULL, category TEXT NOT NULL, amount_minor INTEGER NOT NULL CHECK(amount_minor > 0))")
    with connection:
        connection.executemany(
            "INSERT INTO expenses(day, category, amount_minor) VALUES (?, ?, ?)",
            [(record.day.isoformat(), record.category, record.amount_minor) for record in records],
        )

def category_report(connection):
    return connection.execute("SELECT category, SUM(amount_minor) FROM expenses GROUP BY category ORDER BY category").fetchall()

with TemporaryDirectory() as folder:
    database_path = Path(folder) / "expenses.sqlite"
    with closing(sqlite3.connect(database_path)) as connection:
        save_expenses(connection, expenses)
    with closing(sqlite3.connect(database_path)) as connection:
        report_rows = category_report(connection)
print(report_rows)
assert dict(report_rows) == {"books": 20000, "food": 17000, "travel": 8000}


[('books', 20000), ('food', 17000), ('travel', 8000)]


**Try a change:** Add another valid expense to the CSV and verify the category report changes correctly.


## 3. Export and test failure paths

Export formatted amounts with Decimal to preserve exact decimal values. Test malformed input as well as totals. This project is deliberately small: before turning it into a long-lived application, add duplicate-import handling, schema migration, and a clear currency policy.


In [3]:
def export_report(rows):
    output = io.StringIO(newline="")
    writer = csv.writer(output)
    writer.writerow(["category", "amount"])
    for category, minor in rows:
        writer.writerow([category, f"{Decimal(minor) / 100:.2f}"])
    return output.getvalue()

exported = export_report(report_rows)
print(exported)
assert sum(record.amount_minor for record in expenses) == 45000
assert list(csv.DictReader(io.StringIO(exported)))[0] == {"category": "books", "amount": "200.00"}
for bad_amount in ["NaN", "Infinity", "0", "-1", "1.005", "oops"]:
    try:
        parse_expense({"date": "2025-01-01", "category": "food", "amount": bad_amount})
    except ValueError:
        pass
    else:
        raise AssertionError(f"should reject {bad_amount}")
print("Capstone validation checks passed")


category,amount
books,200.00
food,170.00
travel,80.00

Capstone validation checks passed


**Try a change:** Add missing-field, empty-category, and invalid-date cases to the checks.


## Practice: solve before looking at the answers

Replace the starter code with your implementation. Run it, then change that exercise's `exercise_N_ready` flag to `True` and run its check cell. A failed assertion is feedback: compare the expected value with your result. Add at least one test of your own.


### Exercise 1: Monthly totals

Implement monthly_totals(records) returning a dictionary from YYYY-MM to total integer minor units. Accept Expense objects.

<details>
<summary>Show a hint</summary>

Use the date's strftime('%Y-%m') as the grouping key.

</details>


In [4]:
def monthly_totals(records):
    pass


In [5]:
exercise_1_ready = False  # Set True when your implementation is ready.
if exercise_1_ready:
    assert monthly_totals(expenses) == {"2025-01": 25000, "2025-02": 20000}
    assert monthly_totals([]) == {}
    print("Exercise 1: supplied checks passed. Add your own edge cases!")
else:
    print("Exercise 1: not checked yet. Complete the code, then set exercise_1_ready = True.")


Exercise 1: not checked yet. Complete the code, then set exercise_1_ready = True.


### Exercise 2: Budget alerts

Implement over_budget(totals, budgets). Return sorted categories where spending is strictly greater than the known budget. Ignore categories without a budget; equality is allowed.

<details>
<summary>Show a hint</summary>

Require both a known budget and amount > budget.

</details>


In [6]:
def over_budget(totals, budgets):
    pass


In [7]:
exercise_2_ready = False  # Set True when your implementation is ready.
if exercise_2_ready:
    assert over_budget({"food": 17000, "books": 20000, "travel": 8000}, {"food": 15000, "books": 20000}) == ["food"]
    assert over_budget({}, {}) == []
    assert over_budget({"b": 2, "a": 3}, {"b": 1, "a": 1}) == ["a", "b"]
    print("Exercise 2: supplied checks passed. Add your own edge cases!")
else:
    print("Exercise 2: not checked yet. Complete the code, then set exercise_2_ready = True.")


Exercise 2: not checked yet. Complete the code, then set exercise_2_ready = True.


## Build it yourself

Finish your own version: add a monthly report, budget alerts, a chart, and a rejected-row CSV. Then move domain logic into .py modules and add a command-line interface. Acceptance criteria: 4 accepted and 2 rejected fixture rows; total 450.00; exact category totals; no invalid row stored; parameterized queries; tests pass after a fresh restart. Stretch: prevent duplicate imports using a documented record ID strategy, and explain your tradeoffs.

This open-ended extension has no single supplied solution. Record your design choices and test your result.


In [8]:
# Your mini-project workspace. Add more cells as needed.


## Common mistakes and limits

The database constraint protects positive stored amounts, while the parser enforces the richer input contract. Calling save_expenses twice appends twice: a real import workflow needs an explicit duplicate policy. Completing the course starts your expert practice; build and review larger projects to deepen it.


## Before you move on

- [ ] I can explain the examples without reading the code.
- [ ] I changed at least two inputs and predicted the results.
- [ ] I completed both exercises and enabled their checks.
- [ ] I added a boundary or failure case of my own.
- [ ] I attempted the mini-project and recorded one lesson learned.

**Reflection:** What surprised you? What would you try next? Edit this Markdown cell with your notes.


[Course guide](../README.md) · [Worked solutions](../solutions/23_capstone_expense_tracker_solutions.ipynb) · [Previous notebook](22_performance_and_software_design.ipynb)
